In [1]:
import sys
# Add parent directory to path to import helper_functions module
sys.path.insert(0, '..')

import helper_functions
from constants import *

import eelbrain

SUBJECTS = helper_functions.get_subjects()

# Simple envelope TRFs
Uses the acoustic envelope as the only predictor

In [2]:
def estimate_trfs(predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.FORWARD, padded=False, predictor_dir=MAT_FILE_CONCAT_DIR, trf_dir=MAT_FILE_TRF_DIR):

    # Ensure list of Predictors
    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]

    predictor_names = sorted(predictor_names, key=lambda p: p.value)

    model_name = helper_functions.get_trf_model_name(
        predictor_names, attention, model_type, padded
    )

    suffix = "_padded" if padded else ""

    for subject in SUBJECTS:
        print("-" * 50)
        print(f"Processing {subject} | model: {model_name}")

        subject_trf_dir  = trf_dir / subject
        trf_path = subject_trf_dir / f"{subject}_{model_name}_trf.pickle"

        # --- Skip if exists ---
        if trf_path.exists():
            print("TRF exists, skipping.")
            continue

        # --- Load EEG ---
        eeg_path = EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"

        if not eeg_path.exists():
            print("Missing EEG, skipping.")
            continue

        eeg = eelbrain.load.unpickle(eeg_path)

        # --- Load predictors ---
        predictors = []

        for p in predictor_names:
            predictor_name = helper_functions.get_predictor_name(p, attention, padded)
            predictor_path = predictor_dir / subject / f"{predictor_name}_concat.pickle"

            if not predictor_path.exists():
                print(f"Missing predictor: {predictor_path}, skipping subject.")
                break

            predictors.append(eelbrain.load.unpickle(predictor_path))
        else:
            # --- Prepare x ---
            x = predictors if len(predictors) > 1 else predictors[0]

            print(f"Estimating {model_name} TRF...")

            # --- Lag handling + boosting call ---
            if model_type == MODEL_TYPE.FORWARD:
                tmin, tmax = TRF_LAG_START, TRF_LAG_END
                trf = eelbrain.boosting(
                    eeg,   # y: response
                    x,     # x: predictor
                    tmin, tmax,
                    error='l1', basis=BASIS_FUNCTION_WIDTH,
                    partitions=5, test=1, selective_stopping=True
                )
            else:
                tmin, tmax = -TRF_LAG_END, -TRF_LAG_START
                trf = eelbrain.boosting(
                    x,     # y: response (envelope)
                    eeg,   # x: predictor (EEG)
                    tmin, tmax,
                    error='l1', basis=BASIS_FUNCTION_WIDTH,
                    partitions=5, test=1, selective_stopping=True
                )

            # --- Save ---
            subject_trf_dir.mkdir(exist_ok=True, parents=True)
            eelbrain.save.pickle(trf, trf_path)

            print(f"Saved → {trf_path}")

In [3]:
checks = [
    # Single predictors
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  True,  MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  True,  MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),

    # Combined predictors
    ([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    ([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),

    # Backward models
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    #(PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, True,  MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    #(PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, True,  MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False, MAT_FILE_CONCAT_DIR, MAT_FILE_TRF_DIR),
]

for predictors, attention, model, padded, predictor_dir, trf_dir in checks:
    estimate_trfs(predictors, attention, model, padded, predictor_dir, trf_dir)

print("Done estimating TRFs.")

--------------------------------------------------
Processing S1 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S2 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S3 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S4 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S5 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S6 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S7 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S8 | model: forward_attended_envelope
TRF exists, skipping.
------------------------

In [4]:
# SANITY CHECK: Check dimensions of estimated TRFs
for subject in SUBJECTS:
    print(f"\n{subject}")
    for predictors, attention, model, padded, predictor_dir, trf_dir in checks:
        model_name = helper_functions.get_trf_model_name(predictors, attention, model, padded)
        trf_path = trf_dir / subject / f"{subject}_{model_name}_trf.pickle"
        if trf_path.exists():
            trf = eelbrain.load.unpickle(trf_path)
            print(f"  ✓ {model_name}: {trf.h}")
        else:
            print(f"  ✗ {model_name}: MISSING")


S1
  ✓ forward_attended_envelope: <NDVar 'attended_envelope': 64 sensor, 72 time>
  ✓ forward_ignored_envelope: <NDVar 'ignored_envelope': 64 sensor, 72 time>
  ✓ forward_attended_envelope_padded: <NDVar 'attended_envelope_padded': 64 sensor, 72 time>
  ✓ forward_ignored_envelope_padded: <NDVar 'ignored_envelope_padded': 64 sensor, 72 time>
  ✓ forward_attended_envelope_onset: <NDVar 'attended_envelope_onset': 64 sensor, 72 time>
  ✓ forward_ignored_envelope_onset: <NDVar 'ignored_envelope_onset': 64 sensor, 72 time>
  ✓ forward_attended_envelope+envelope_onset: (<NDVar 'attended_envelope': 64 sensor, 72 time>, <NDVar 'attended_envelope_onset': 64 sensor, 72 time>)
  ✓ forward_ignored_envelope+envelope_onset: (<NDVar 'ignored_envelope': 64 sensor, 72 time>, <NDVar 'ignored_envelope_onset': 64 sensor, 72 time>)
  ✓ backward_attended_envelope: <NDVar: 64 sensor, 73 time>
  ✓ backward_ignored_envelope: <NDVar: 64 sensor, 73 time>
  ✓ backward_attended_envelope_onset: <NDVar: 64 sensor, 7

In [5]:
def delete_trfs(predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.FORWARD, padded=False, trf_dir=MAT_FILE_TRF_DIR):
    """
    Delete TRF files across all subjects so they can be recomputed.

    Args:
        predictor_names: A single PREDICTOR_TYPE or list of PREDICTOR_TYPE values.
        attention:       ATTENTION_TYPE.ATTENDED or ATTENTION_TYPE.IGNORED.
        model_type:      MODEL_TYPE.FORWARD or MODEL_TYPE.BACKWARD.
        padded:          Whether the TRF was estimated with padded data.
        trf_dir:         Directory where TRFs are stored.
    """
    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]

    model_name = helper_functions.get_trf_model_name(predictor_names, attention, model_type, padded)

    deleted = 0
    missing = 0

    for subject in SUBJECTS:
        trf_path = trf_dir / subject / f"{subject}_{model_name}_trf.pickle"
        if trf_path.exists():
            trf_path.unlink()
            print(f"  ✓ Deleted {trf_path}")
            deleted += 1
        else:
            print(f"  - Not found: {trf_path}")
            missing += 1

    print(f"\nDone — {deleted} deleted, {missing} not found.")

In [ ]:
# CAUTION: Irreversible deletion! Use with care.
# example delete_trfs(PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD)
